# 01 — Merge Raw Data

**Goal:** combine the three per-year post-monsoon CSVs into a single table — *nothing else*.

The three raw files describe the same measurements but use different column
*names* (`E.C` vs `EC`, `CO3` vs `CO_-2`, `Cl` vs `Cl -`, ...) and the 2020 file
carries a stray empty column. A naive `pd.concat` would therefore create
duplicate/misaligned columns. So the only operations here are **structural
alignment** so the rows can stack:

1. strip whitespace from headers and rename to one common schema
2. drop the empty `Unnamed` column (2020)
3. add an explicit `year` column
4. `concat` the three frames

We deliberately do **not** coerce types, normalise `season`, impute, drop
duplicates, or derive any labels. All of that lives in `02_Data_Cleaning`.

**Output:** `data/interim/groundwater_merged_2018_2020.csv`

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    2018: RAW_DIR / "ground_water_quality_2018_post.csv",
    2019: RAW_DIR / "ground_water_quality_2019_post.csv",
    2020: RAW_DIR / "ground_water_quality_2020_post.csv",
}
RAW_FILES

{2018: PosixPath('/tmp/gwra/data/raw/ground_water_quality_2018_post.csv'),
 2019: PosixPath('/tmp/gwra/data/raw/ground_water_quality_2019_post.csv'),
 2020: PosixPath('/tmp/gwra/data/raw/ground_water_quality_2020_post.csv')}

## Canonical column map

Every raw header (after `.strip()`) maps to one clean name. Both spelling variants of each measurement point to the same target.

In [ ]:
COLUMN_MAP = {
    "sno": "sno", "district": "district", "mandal": "mandal", "village": "village",
    "lat_gis": "lat", "long_gis": "lon", "gwl": "gwl", "season": "season",
    "pH": "ph",
    "E.C": "ec", "EC": "ec",
    "TDS": "tds",
    "CO3": "co3", "CO_-2": "co3",
    "HCO3": "hco3", "HCO_ -": "hco3",
    "Cl": "cl", "Cl -": "cl",
    "F": "f", "F -": "f",
    "NO3": "no3", "NO3-": "no3",
    "SO4": "so4", "SO4-2": "so4",
    "Na": "na", "Na+": "na",
    "K": "k", "K+": "k",
    "Ca": "ca", "Ca+2": "ca",
    "Mg": "mg", "Mg+2": "mg",
    "T.H": "th", "SAR": "sar",
    "Classification": "irrigation_class",
    "RSC  meq  / L": "rsc",
    "Classification.1": "rsc_class",
}

## Load + structurally align each file

Read raw, drop `Unnamed` junk, strip headers, rename, tag `year`. Values are left exactly as read.

In [ ]:
def load_aligned(path, year):
    df = pd.read_csv(path, encoding="latin-1")
    # drop the stray empty column (2020's "Unnamed: 8")
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])
    # strip header whitespace, then rename to the common schema
    df.columns = [str(c).strip() for c in df.columns]
    unknown = [c for c in df.columns if c not in COLUMN_MAP]
    if unknown:
        raise ValueError(f"[{year}] unmapped columns: {unknown}")
    df = df.rename(columns=COLUMN_MAP)
    df["year"] = year          # provenance only; no value cleaning here
    return df

frames = {yr: load_aligned(p, yr) for yr, p in RAW_FILES.items()}
for yr, df in frames.items():
    print(f"{yr}: {df.shape[0]} rows, {df.shape[1]} cols")

2018: 374 rows, 27 cols
2019: 364 rows, 27 cols
2020: 368 rows, 27 cols


### Confirm schemas now match

After renaming, all three frames should share an identical column set — the precondition for a clean stack.

In [ ]:
cols = [set(df.columns) for df in frames.values()]
assert cols[0] == cols[1] == cols[2], "schemas still differ!"
print("Schemas aligned:", sorted(cols[0]))

Schemas aligned: ['ca', 'cl', 'co3', 'district', 'ec', 'f', 'gwl', 'hco3', 'irrigation_class', 'k', 'lat', 'lon', 'mandal', 'mg', 'na', 'no3', 'ph', 'rsc', 'rsc_class', 'sar', 'season', 'sno', 'so4', 'tds', 'th', 'village', 'year']


## Concatenate

In [ ]:
merged = pd.concat(frames.values(), ignore_index=True)
print("Merged shape:", merged.shape)
print("Rows per year:")
print(merged["year"].value_counts().sort_index().to_string())
merged.head()

Merged shape: (1106, 27)
Rows per year:
year
2018    374
2019    364
2020    368


,sno,district,mandal,village,lat,lon,gwl,season,ph,ec,...,na,k,ca,mg,th,sar,irrigation_class,rsc,rsc_class,year
0,1,ADILABAD,Adilabad,Adilabad,19.668300,78.524700,5.09,postmonsoon 2018,8.28,745,...,49.0,4.0,48.0,38.896,279.934211,1.273328,C2S1,-1.198684,P.S.,2018
1,2,ADILABAD,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,postmonsoon 2018,8.29,921,...,42.0,5.0,56.0,63.206,399.893092,0.913166,C3S1,-3.397862,P.S.,2018
2,3,ADILABAD,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,postmonsoon 2018,7.69,510,...,45.0,2.0,24.0,38.896,219.934211,1.319284,C2S1,-0.398684,P.S.,2018
3,4,ADILABAD,Jainath,Jainath,19.730555,78.640000,5.75,postmonsoon 2018,8.09,422,...,27.0,1.0,32.0,19.448,159.967105,0.928155,C2S1,0.000658,P.S.,2018
4,5,ADILABAD,Narnoor,Narnoor,19.495665,78.852654,2.15,postmonsoon 2018,8.21,2321,...,298.0,5.0,56.0,92.378,519.843750,5.682664,C4S2,-4.396875,P.S.,2018


## Save merged (interim)

Still raw values — just stacked. Cleaning happens next.

In [ ]:
out = INTERIM_DIR / "groundwater_merged_2018_2020.csv"
merged.to_csv(out, index=False)
print("Wrote", out.relative_to(ROOT), "|", merged.shape)

Wrote data/interim/groundwater_merged_2018_2020.csv | (1106, 27)
